# NB_10: Đóng Gói Reproducibility Package Một Chạm & Kiểm Tra 4 Cửa Ải Nộp Bài ScholarOne IEEE TPAMI
**Dự án mục tiêu:** *Complex Wasserstein State-Space Network (CW-SSN)* — Nộp tạp chí Q1 **IEEE TPAMI**  
**Giai đoạn:** Giai đoạn 5 (Tuần 13 – Tuần 16) | **Phân bổ phần cứng:** 1x Tài khoản Kaggle T4x2 hoặc Colab Pro | **Thời gian phiên:** 1 – 2 giờ

---

## 1. Mục Tiêu Chiến Lược & Gatekeeping KPIs
1. **Thực thi kiểm thử một chạm (One-Click Reproducibility Audit)** trên toàn bộ mã nguồn `models/cw_ssn.py`, `tests/test_cw_ssn.py`, và cơ sở dữ liệu `TPAMI_PreResearch_References_25Papers.csv` trong vòng **dưới 15 phút** với sai số **0.00 dB**.
2. **Kiểm tra tự động 4 tiêu chí bắt buộc của Ban Biên tập (Associate Editor) IEEE TPAMI** trên bản thảo `paper/main_tpami.tex` và thư ngỏ `paper/cover_letter_tpami.tex`:
   - **Gate 1 (Định vị phạm vi TPAMI vs. IEEE TIP):** Nêu bật ngay trong 2 câu mở đầu Cover Letter rằng bài báo đề xuất toán tử nguyên thủy mới (`CW-SSO`) cho *Pattern Analysis & Machine Intelligence*, được bảo chứng bởi 2 Định lý toán học và kiểm chứng xuyên suốt từ Khôi phục tín hiệu (3D/5D/10D AiOIR) đến Phân đoạn cấu trúc tiền cảnh phổ quát (EVP).
   - **Gate 2 (Định dạng Regular Paper 14 trang đôi):** Sử dụng `\documentclass[10pt,journal,compsoc]{IEEEtran}`, toàn bộ chứng minh chi tiết các Bổ đề (Lemmas) được tách rõ cho phần Supplementary Material.
   - **Gate 3 (Danh sách 3–5 chuyên gia phản biện độc lập — Conflict-Free Reviewers):** Đề xuất các chuyên gia đầu ngành về Optimal Transport IR và Vision State-Space Models kèm lý do chuyên môn.
   - **Gate 4 (Tuyên bố nguyên bản bắt buộc):** Cam kết tính nguyên bản (Originality), không nộp kép (Not under review elsewhere), và sự đồng thuận của toàn bộ tác giả.

In [ ]:
import os
import sys
import time
import math
import json
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure repository root is on sys.path (works on Kaggle, Colab Pro, and Local)
REPO_URL = "https://github.com/hannhu11/Complex-Wasserstein-State-Space-Network-CW-SSN.git"
REPO_DIR = Path("Complex-Wasserstein-State-Space-Network-CW-SSN")

if not Path("models/cw_ssn.py").exists():
    if REPO_DIR.exists() and (REPO_DIR / "models/cw_ssn.py").exists():
        sys.path.insert(0, str(REPO_DIR.resolve()))
    elif Path("..").resolve().joinpath("models/cw_ssn.py").exists():
        sys.path.insert(0, str(Path("..").resolve()))
    else:
        print(f"[Setup] Cloning {REPO_URL} ...")
        os.system(f"git clone --depth 1 {REPO_URL}")
        if REPO_DIR.exists():
            sys.path.insert(0, str(REPO_DIR.resolve()))
else:
    sys.path.insert(0, str(Path(".").resolve()))

from models.cw_ssn import (
    ComplexWassersteinSSM2D,
    CWSSBlock,
    CWSSN,
    CWSSOAdapter,
    SpectralWassersteinLoss,
    build_cw_ssn_tiny,
    build_cw_ssn_base,
    rotational_2x2_Scan_parallel,
    rotational_2x2_scan_sequential,
)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Environment] PyTorch: {torch.__version__} | Device: {DEVICE} | GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(idx)
        print(f"  -> GPU {idx}: {p.name} ({p.total_memory / (1024**3):.2f} GB VRAM)")

## 2. Chạy Kiểm Thử Toàn Diện Mã Nguồn, Bộ Test Toán Học & Kiểm Duyệt Hồ Sơ Nộp Bài TPAMI

In [ ]:
import unittest
import csv
import importlib.util

t0 = time.perf_counter()

# 1. Run the complete unit test suite programmatically
test_candidates = [
    Path("tests/test_cw_ssn.py"),
    Path("../tests/test_cw_ssn.py"),
    REPO_DIR / "tests/test_cw_ssn.py",
]
test_path = next((p for p in test_candidates if p.exists()), None)
assert test_path is not None, "tests/test_cw_ssn.py not found!"
spec = importlib.util.spec_from_file_location("test_cw_ssn", test_path)
test_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(test_mod)

suite = unittest.defaultTestLoader.loadTestsFromTestCase(test_mod.TestCWSSNArchitectureAndMath)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
assert result.wasSuccessful(), "Unit test suite failed!"

# 2. Verify the 25-Paper 15-Column Reference CSV
csv_candidates = [
    Path("TPAMI_PreResearch_References_25Papers.csv"),
    Path("../TPAMI_PreResearch_References_25Papers.csv"),
    REPO_DIR / "TPAMI_PreResearch_References_25Papers.csv",
]
csv_path = next((p for p in csv_candidates if p.exists()), None)
assert csv_path is not None, "TPAMI_PreResearch_References_25Papers.csv not found!"
with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
    rows = list(csv.reader(f))
assert len(rows) == 26 and all(len(r) == 15 for r in rows), "CSV structure mismatch!"
print(f"[Reproducibility Check] Verified {csv_path}: {len(rows)-1} papers x {len(rows[0])} columns.")

# 3. Audit the 4 Mandatory IEEE TPAMI ScholarOne Gates in paper/main_tpami.tex and paper/cover_letter_tpami.tex
tex_candidates = [Path("paper"), Path("../paper"), REPO_DIR / "paper"]
paper_dir = next((p for p in tex_candidates if (p / "main_tpami.tex").exists()), None)
assert paper_dir is not None, "paper/ directory not found!"

main_tex = (paper_dir / "main_tpami.tex").read_text(encoding="utf-8")
cover_tex = (paper_dir / "cover_letter_tpami.tex").read_text(encoding="utf-8")

assert "\documentclass[10pt,journal,compsoc]{IEEEtran}" in main_tex, "Gate 2 Failed: Missing IEEEtran compsoc header!"
assert "Theorem" in main_tex and "Supplementary Material" in main_tex, "Gate 2 Failed: Missing Theorems or Supplementary Material reference!"
assert "IEEE TIP" in cover_tex and "Pattern Analysis" in cover_tex, "Gate 1 Failed: Missing explicit TPAMI vs IEEE TIP positioning!"
assert "Suggested Conflict-Free Reviewers" in cover_tex, "Gate 3 Failed: Missing Conflict-Free Reviewers list!"
assert "not currently under consideration" in cover_tex and "All authors have read and approved" in cover_tex, "Gate 4 Failed: Missing mandatory originality/consent declarations!"
print("[TPAMI Editorial Check] All 4 ScholarOne Submission Gates verified in main_tpami.tex and cover_letter_tpami.tex.")

elapsed = time.perf_counter() - t0
print(f"[KPI Check] Complete Reproducibility Verification finished in {elapsed:.2f} seconds (Target: < 900s).")
assert elapsed < 900.0
print("[GATEKEEPER PASSED] Ready for IEEE TPAMI ScholarOne Manuscript Submission.")